# **Understanding Data Types in Python**

The fundamental difference between Python and languages like C or Java lies in **when** the language verifies the types of your variables and how it allocates memory. This is often summarized as "Dynamic Typing" versus "Static Typing."

---

- **1. Static Typing (C/Java): The "Strict Blueprint"**
    - In languages like C or Java, you must declare the data type of every variable before you use it. The compiler checks these types *before* the program even runs.
        * **Fixed Types:** Once you define `int x = 10;`, that variable `x` can only ever hold an integer. If you try to assign `x = "hello";`, the program will refuse to compile.
        * **Compile-Time Verification:** The compiler acts as a strict guard. It catches type-related errors (like trying to perform math on a string) before the code is executed.
    * **Performance:** Because the compiler knows exactly what type of data each variable holds, it can optimize the machine code effectively. It knows exactly how many bytes of memory an `int` or a `float` needs.


- **2. Dynamic Typing (Python): The "Flexible Label"**
    - In Python, you do not declare types. You simply assign values to names.
        * **Type Labels, Not Boxes:** In Python, a variable is just a "label" (or reference) pointing to an object in memory. You can point the label `x` to an integer, and then immediately point that same label to a string.
        * **Runtime Verification:** Python checks types only when the code actually executes. If you try to perform `5 + "hello"`, Python will throw a `TypeError` **only when the program reaches that specific line**.
        * **Flexibility:** This makes prototyping very fast. You don't have to write boilerplate code for type declarations, which is why Python is favored for data exploration and rapid development.

---

- **Comparison at a Glance**

| Feature | Static (C/Java) | Dynamic (Python) |
| :--- | :--- | :--- |
| **Type Checking** | Compile-time (Before running) | Runtime (During running) |
| **Variable Declaration** | Mandatory | Not required |
| **Flexibility** | Rigid | Highly flexible |
| **Error Catching** | Early (at build time) | Later (at runtime) |
| **Typical Goal** | Performance & Scalability | Speed of Development & Flexibility |

---

- **Why does this matter for your Data Engineering projects?**
    - Since you are working with the **Medallion Architecture**, you are handling various data types (JSONs, Parquet files, CSVs) in your Bronze, Silver, and Gold layers.

* **The Python Trade-off:** In your ETL pipelines, the dynamic nature of Python allows you to write generic functions that can handle different data structures without re-writing code. However, it also means you are more vulnerable to "hidden" runtime errors—like a column that should be an integer suddenly containing strings—that won't be caught until your pipeline crashes during a run.
* **The C/Java Trade-off:** If you were building a massive, mission-critical data processing engine (like Apache Spark, which is written in Scala/Java), static typing is a massive advantage. It ensures that data schemas are locked down and provides the performance required to process petabytes of data efficiently.


> As a Data Engineer, you are essentially using **Python** to orchestrate the high-level flow (flexibility) while relying on underlying **C** or **Java-based** libraries (like Pandas, NumPy, or Spark) to handle the heavy, performance-critical data processing.


## **A Python Integer Is More Than Just an Integer**

To understand how Python and C store an integer, you have to look at the difference between **Value Types** (in C) and **Object Types** (in Python).

- **1. C: The "Fixed Box" (Static)**
    - In C, a `long` variable is a direct map to a specific block of memory. 
        * **Fixed Allocation:** When you declare `long x = 100;`, the C compiler knows exactly how many bytes a `long` requires (typically 8 bytes on a 64-bit system).
        * **Direct Access:** The computer allocates 8 contiguous bytes in memory. The value `100` is written directly into those 8 bytes in binary format.
        * **No Metadata:** The memory contains **only** the value. The program knows it is a `long` because of the variable type defined at compile-time. If you tell the compiler to treat those 8 bytes as a pointer or a double, it will do so blindly—which is why C is fast but dangerous.

---

- **2. Python: The "Object Box" (Dynamic)**
    - In Python, everything is an **object**. Even a simple integer is a complex structure stored on the "heap."
        * **The Variable as a Label:** A Python variable like `x = 100` is not a storage box; it is a **reference** (a pointer) to an object.
        * **The Object Structure:** Because Python must support arbitrary-precision integers (they never overflow, unlike C's `long`), an integer object contains metadata. It is a structure that includes:
            * **Reference count:** Used for memory management (garbage collection).
            * **Type information:** A pointer to the `int` type object (so Python knows it's an integer).
            * **Size information:** How many digits the integer has.
            * **The actual value:** The data itself.
    * **Overhead:** Because of this extra metadata, a simple Python integer consumes significantly more memory (often 28 bytes or more) than the 8 bytes used by a C `long`.

![he difference between C and Python integers](../Images/The_difference_between_C_and_Python_integers.png)

---

- **Comparison Summary**

| Feature | C `long` | Python `int` |
| :--- | :--- | :--- |
| **Size** | Fixed (e.g., 8 bytes) | Dynamic (at least 28 bytes) |
| **Storage** | Raw binary value | Metadata + value |
| **Type Tracking** | None (at runtime) | Stored within the object |
| **Flexibility** | Rigid | Handles arbitrarily large numbers |

---

- **Why this matters for your work**
    - In your **Medallion architecture** project, you are dealing with large datasets:
        1.  **Efficiency:** When you use libraries like **NumPy** or **Pandas**, you are essentially forcing Python to use C-like "fixed boxes" (arrays) under the hood. This is why NumPy is thousands of times faster than standard Python lists—it bypasses the "object" overhead and stores raw integers in contiguous memory just like C.
        2.  **Safety:** Python’s dynamic object structure prevents the integer overflow errors common in C, but the "price" you pay for that safety is higher memory consumption per variable.


## **A Python List Is More Than Just a List**

The difference between a standard Python `list` and a NumPy `array` is one of the most critical concepts for a Data Engineer. It is the difference between a flexible container for "anything" and a high-performance engine for "data."

---

- **1. Python `list`: The Dynamic Container**
    - A Python list is a **dynamic array of references**. It is designed for maximum flexibility, not for performance.

* **Heterogeneous:** A single list can hold an integer, a string, and a dictionary at the same time.
* **Pointer Overhead:** Because the list doesn't know what types it contains, it stores a list of pointers (references) to objects scattered throughout your computer's memory.
* **Memory Inefficiency:** Every item in the list is a full-blown Python object with its own metadata (reference count, type info, etc.).
* **Computational Cost:** To perform an operation (like adding 1 to every element), Python must:
    1.  Follow the pointer to the object.
    2.  Check the type of the object.
    3.  Find the addition method for that type.
    4.  Execute the addition and create a new object.

---

- **2. NumPy `array`: The Fixed-Type Powerhouse**
    - A NumPy array (`ndarray`) is a **block of contiguous memory** designed for fast numerical processing.

* **Homogeneous:** Every element in a NumPy array **must** be the same data type (e.g., `int64`, `float64`).
* **Memory Efficiency:** Because the type is fixed, NumPy doesn't need to store metadata for every single element. It stores the raw data in a solid block of memory.
* **CPU Cache Friendly:** Because the data is contiguous (side-by-side in RAM), your CPU can load chunks of the array into its high-speed cache efficiently. This is the secret behind NumPy's speed.
* **Vectorization:** NumPy operations use "Vectorization." Instead of looping through elements one by one, it performs operations on the entire block of memory at once, often using low-level C and Fortran code that talks directly to your CPU's hardware.

---

![The difference between C and Python lists](../Images/The_difference_between_C_and_Python_lists.png)

- **Comparison Summary**

| Feature | Python `list` | NumPy `array` |
| :--- | :--- | :--- |
| **Data Types** | Heterogeneous (Mixed) | Homogeneous (Same) |
| **Storage** | Pointers to objects (Scattered) | Contiguous memory (Fixed) |
| **Performance** | Slow (due to type checking) | Fast (due to vectorization) |
| **Flexibility** | High (insert/delete anywhere) | Low (fixed size and type) |
| **Memory usage** | High (due to metadata) | Low (raw data only) |

---

- **Why this matters for your `Comptoire-DWH`**
    - In your data warehousing project, your performance depends entirely on how you handle your ETL (Extract, Transform, Load) operations:
        * **Use `lists`** when you are handling small configuration files, metadata dictionaries, or when you need to dynamically grow a structure that contains different types of information.
        * **Use `NumPy arrays` (or Pandas, which is built on them)** when you are processing the **Silver** or **Gold** layer data. If you are applying transformations to millions of rows of sales data, a Python `list` will be the bottleneck that makes your pipeline crawl, whereas a NumPy array will process it almost instantaneously.

## **Fixed-Type Arrays in Python**

Standard Python lists are dynamic and heterogeneous, as we discussed. However, Python provides a built-in module called **`array`** that offers a primitive, fixed-type array implementation. It is often overlooked because NumPy is so powerful, but it is technically a "built-in" way to handle fixed-type data without installing external packages.

---

- **The `array` Module: Python's Built-in Fixed-Type Array**
    - The `array` module stores data in a more compact, C-like format. You must specify a **type code** when you create one, which tells Python exactly how many bytes to allocate for each element.

- **How to use it:**
```python
import array

# 'i' stands for signed integer (usually 4 bytes)
arr = array.array('i', [10, 20, 30, 40])

# This will raise a TypeError because it's fixed-type
# arr.append("not an integer") 
```

---

- **Comparison: Python `list` vs. `array` vs. `NumPy`**

| Feature | Python `list` | Built-in `array` | NumPy `array` |
| :--- | :--- | :--- | :--- |
| **Type** | Dynamic/Mixed | Fixed (defined at creation) | Fixed (defined at creation) |
| **Memory** | High (Pointers) | Low (Raw C-style) | Very Low (Contiguous) |
| **Operations** | Slow (General) | Basic | Fast (Vectorized) |
| **Requirement** | Built-in | Built-in module | Requires installation |

---

- **Why use the `array` module instead of NumPy?**
    - While NumPy is the gold standard for data engineering, the built-in `array` module is useful in specific, niche scenarios:
        1.  **Lightweight requirements:** You are writing a script that cannot have external dependencies (like `numpy`).
        2.  **Memory constraints:** You have a very large sequence of simple integers or floats and need to save memory compared to a standard `list`, but don't need the heavy overhead of the entire NumPy library.
        3.  **Basic storage:** You need to store binary data that maps directly to C data structures for communication with a low-level API.

- **Important Limitation**
    - The built-in `array` module **does not support vectorization.** Even though the data is stored efficiently (like a NumPy array), if you want to add 1 to every element, you still have to use a Python `for` loop, which is slow. NumPy’s true power is its ability to perform math on the *entire block* of memory at the CPU level.

> For your **Medallion architecture** project, you will almost certainly prefer **NumPy** for any data processing because it allows you to perform complex transformations on millions of rows at C-speed, whereas the built-in `array` module would still require slow Python-level iteration.

In [1]:
import array

In [2]:
L = [i for i in range(10)]
arr = array.array('i', L)

arr

array('i', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [3]:
arr.append(1)

In [4]:
arr

array('i', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 1])

## **Creating Arrays from Python Lists**

In [10]:
import numpy as np

In [14]:
a = np.array([1, 2, 3, 4, 5])

a

array([1, 2, 3, 4, 5])

In [13]:
# numpy upcats the type in uniform type according to the type promotion rules
b = np.array([1.2, 2, 3, 4, 5.17])

b

array([1.2 , 2.  , 3.  , 4.  , 5.17])

In [16]:
# define explicitly the data type of values
a = np.array([1, 2, 3, 4, 5], dtype=np.float32)

a

array([1., 2., 3., 4., 5.], dtype=float32)

In [17]:
# Nested lists result in multidimensional arrays
b = np.array([range(i, i+3) for i in [2, 4, 6]])

b

array([[2, 3, 4],
       [4, 5, 6],
       [6, 7, 8]])

> The inner lists are treated as **rows** of the resulting two-dimensional array.

## **Creating Arrays from Scratch**

In [18]:
# Create a length-10 integer array filled with 0s

np.zeros(10, dtype=np.int8)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)

In [19]:
# Create a 3x5 floating-point array filled with 1s

np.ones((3, 5), dtype=np.int8)

array([[1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1]], dtype=int8)

In [20]:
# Create a 3x5 array filled with 3.14

np.full((3, 5), fill_value=3.14, dtype=np.float16)

array([[3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14]], dtype=float16)

In [23]:
# Create an array filled with a linear sequence
# starting at 0, ending at 20, stepping by 2
# (this is similar to the built-in range function)

np.arange(0, 20, 2)

array([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])

In [26]:
# Create an array of five values evenly spaced between 0 and 1

np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [28]:
# Create a 3x3 array of uniformly distributed
# pseudorandom values between 0 and 1

np.random.random(size=(3, 3))

array([[0.24588923, 0.78500281, 0.02922493],
       [0.30647426, 0.03940002, 0.29479793],
       [0.90483863, 0.03809542, 0.80269574]])

In [29]:
# Create a 3x3 array of normally distributed pseudorandom
# values with mean 0 and standard deviation 1

np.random.normal(0, 1, size=(3, 3))

array([[ 1.30286221,  0.13618128,  1.18219448],
       [ 0.20777218,  0.35041344,  0.33138945],
       [ 0.29348422, -0.24460001, -0.48031944]])

In [30]:
# Create a 3x3 array of pseudorandom integers in the interval [0, 10)

np.random.randint(0, 10, size=(3, 3))

array([[5, 0, 6],
       [0, 1, 8],
       [9, 4, 8]])

In [31]:
# Create a 3x3 identity matrix

np.eye(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [32]:
# Create an uninitialized array of three integers; the values will be
# whatever happens to already exist at that memory location

np.empty(3)

array([1., 1., 1.])

## **NumPy Standard Data Types**

| Data type | Description |
| :--- | :--- |
| **bool_** | Boolean (True or False) stored as a byte |
| **int_** | Default integer type (same as C long; normally either int64 or int32) |
| **intc** | Identical to C int (normally int32 or int64) |
| **intp** | Integer used for indexing (same as C ssize_t; normally either int32 or int64) |
| **int8** | Byte (–128 to 127) |
| **int16** | Integer (–32768 to 32767) |
| **int32** | Integer (–2147483648 to 2147483647) |
| **int64** | Integer (–9223372036854775808 to 9223372036854775807) |
| **uint8** | Unsigned integer (0 to 255) |
| **uint16** | Unsigned integer (0 to 65535) |
| **uint32** | Unsigned integer (0 to 4294967295) |
| **uint64** | Unsigned integer (0 to 18446744073709551615) |
| **float_** | Shorthand for float64 |
| **float16** | Half-precision float: sign bit, 5 bits exponent, 10 bits mantissa |
| **float32** | Single-precision float: sign bit, 8 bits exponent, 23 bits mantissa |
| **float64** | Double-precision float: sign bit, 11 bits exponent, 52 bits mantissa |
| **complex_** | Shorthand for complex128 |
| **complex64** | Complex number, represented by two 32-bit floats |
| **complex128** | Complex number, represented by two 64-bit floats |

# **The Basics of NumPy Arrays**

## **NumPy Array Attributes**

In [1]:
import numpy as np

In [107]:
rng = np.random.default_rng(seed=1071) # # seed for reproducibility

x1 = rng.integers(10, size=10)        # one-dimensional array
x2 = rng.integers(10, size=(3, 4))    # two-dimensional array
x3 = rng.integers(10, size=(3, 4, 5)) # three-dimensional array

In [108]:
print("the number of dimension : ", x3.ndim)
print("the shape of array : ", x3.shape)
print("The total of element in array : ", x3.size)
print("the data type of elements in array : ", x3.dtype)

the number of dimension :  3
the shape of array :  (3, 4, 5)
The total of element in array :  60
the data type of elements in array :  int64


## **Array Indexing: Accessing Single Elements**

In [109]:
x1

array([7, 2, 1, 8, 3, 2, 9, 9, 0, 3])

In [110]:
x1[::2]

array([7, 1, 3, 9, 0])

In [111]:
x1[1::2]

array([2, 8, 2, 9, 3])

In [112]:
x1[-10:-1]

array([7, 2, 1, 8, 3, 2, 9, 9, 0])

In [113]:
x1[::-1]

array([3, 0, 9, 9, 2, 3, 8, 1, 2, 7])

In [114]:
x1[-10:-1:-1]

array([], dtype=int64)

- **Start:** Index -10.
- **Stop:** Index -1 (exclusive).
- **Step:** -1.
- **Direction:** When the step is negative, the slice must move from right to left (decreasing indices).
- **The Conflict:** You are starting at index -10 and trying to move to the right (towards index -1) by stepping negatively (-1). This is logically impossible because your start point is to the left of your stop point, but your direction is pointing to the left.

- The Rule of Slicing
    - In Python slicing:
        - If **step > 0**, the start must be less than the stop.
        - If **step < 0**, the start must be greater than the stop.

In [115]:
x1[-1:-100:-1] # we can set the ens of slacing to the out of range bu this don't cause error

array([3, 0, 9, 9, 2, 3, 8, 1, 2, 7])

In [116]:
x1[0:100] # also in positive case

array([7, 2, 1, 8, 3, 2, 9, 9, 0, 3])

In [117]:
x2

array([[8, 8, 6, 1],
       [2, 3, 5, 8],
       [6, 4, 6, 5]])

In [118]:
x2[0, 0] # slicing by (rows, columns)

np.int64(8)

In [119]:
x2[0, -1]

np.int64(1)

In [120]:
x2[:, 1:3]

array([[8, 6],
       [3, 5],
       [4, 6]])

In [121]:
x2[1, 1:3] = 8, 8

x2

array([[8, 8, 6, 1],
       [2, 8, 8, 8],
       [6, 4, 6, 5]])

In [122]:
x2[0, 0] = 3.3145 # # this will be truncated to ensure the Heterogeneity

x2

array([[3, 8, 6, 1],
       [2, 8, 8, 8],
       [6, 4, 6, 5]])

## **Array Slicing: Accessing Subarrays**

### **One-Dimensional Subarrays**

In [123]:
x1

array([7, 2, 1, 8, 3, 2, 9, 9, 0, 3])

In [124]:
x1[::2]

array([7, 1, 3, 9, 0])

In [125]:
x1[1::2]

array([2, 8, 2, 9, 3])

In [126]:
x1[-1:-6:-1]

array([3, 0, 9, 9, 2])

In [127]:
x1[4::-1]

array([3, 8, 1, 2, 7])

### **Multidimensional Subarrays**

In [128]:
x2

array([[3, 8, 6, 1],
       [2, 8, 8, 8],
       [6, 4, 6, 5]])

In [129]:
x2[:, ::2]

array([[3, 6],
       [2, 8],
       [6, 6]])

In [130]:
x2[::-1, ::-1]

array([[5, 6, 4, 6],
       [8, 8, 8, 2],
       [1, 6, 8, 3]])

In [131]:
x2[:, 0]

array([3, 2, 6])

In [132]:
x2[0]

array([3, 8, 6, 1])

In [133]:
x2[:, 1]

array([8, 8, 4])

In [134]:
x2[::-1, 1]

array([4, 8, 8])

### **Subarrays as No-Copy Views**

> Unlike Python list slices, NumPy array slices are returned as **views** rather than copies of the array data.

In [135]:
x2

array([[3, 8, 6, 1],
       [2, 8, 8, 8],
       [6, 4, 6, 5]])

In [136]:
x2_sub = x2[:2, 1:3]

x2_sub

array([[8, 6],
       [8, 8]])

In [137]:
x2_sub[0, 0] = 88

x2

array([[ 3, 88,  6,  1],
       [ 2,  8,  8,  8],
       [ 6,  4,  6,  5]])

In [138]:
x2_sub[1, :] = 88

x2

array([[ 3, 88,  6,  1],
       [ 2, 88, 88,  8],
       [ 6,  4,  6,  5]])

In [139]:
x2_sub[1, :] = [88, 99]

x2

array([[ 3, 88,  6,  1],
       [ 2, 88, 99,  8],
       [ 6,  4,  6,  5]])

In [140]:
x2_sub

array([[88,  6],
       [88, 99]])

> but it can be advantageous: for example, when working with large datasets, we can access and process pieces of these datasets without the need to copy the underlying data buffer.

### **Creating Copies of Arrays**

In [141]:
x2

array([[ 3, 88,  6,  1],
       [ 2, 88, 99,  8],
       [ 6,  4,  6,  5]])

In [142]:
x2_sub_copy = x2[:2, 1:3].copy()

x2_sub_copy

array([[88,  6],
       [88, 99]])

> If we now modify this subarray, the original array is not touched

In [143]:
x2_sub_copy[1, :] = [8, 9]

x2

array([[ 3, 88,  6,  1],
       [ 2, 88, 99,  8],
       [ 6,  4,  6,  5]])

In [144]:
x2_sub_copy

array([[88,  6],
       [ 8,  9]])

## **Reshaping of Arrays**

In [157]:
grid = np.arange(1, 10).reshape((3, 3))

grid

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [158]:
grid.reshape((1, -1))

grid

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [159]:
grid.reshape((-1))

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [160]:
grid.reshape((1, -1))

array([[1, 2, 3, 4, 5, 6, 7, 8, 9]])

In [161]:
x = np.array([1, 2, 3])

x

array([1, 2, 3])

In [162]:
x.reshape((1, 3))

array([[1, 2, 3]])

In [163]:
x.reshape((3, 1))

array([[1],
       [2],
       [3]])

In [164]:
x[np.newaxis, :] # row vector via newaxis

array([[1, 2, 3]])

In [165]:
x[:, np.newaxis] # column vector via newaxis

array([[1],
       [2],
       [3]])

## **Array Concatenation and Splitting**

### **Concatenation of Arrays**

In [166]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

np.concatenate([x, y])

array([1, 2, 3, 4, 5, 6])

In [168]:
z = np.array([7, 8, 9])

np.concatenate([x, y, z])

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [169]:
grid

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [171]:
grid2 = np.arange(1, 10).reshape((3, 3))

grid2

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [173]:
np.concatenate([grid, grid2]) # concatenate along the first axis (rows)

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9],
       [1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [175]:
np.concatenate([grid, grid2], axis=1) # concatenate along the second axis (columns)

array([[1, 2, 3, 1, 2, 3],
       [4, 5, 6, 4, 5, 6],
       [7, 8, 9, 7, 8, 9]])

> For working with arrays of mixed dimensions, it can be clearer to use the **np.vstack** **(vertical stack)** and **np.hstack (horizontal stack)** functions 

In [179]:
x

array([1, 2, 3])

In [177]:
x.ndim, grid.ndim

(1, 2)

In [178]:
np.vstack([x, grid])

array([[1, 2, 3],
       [1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [181]:
y = np.arange(1, 4).reshape((3, 1))

y

array([[1],
       [2],
       [3]])

In [182]:
np.hstack([y, grid])

array([[1, 1, 2, 3],
       [2, 4, 5, 6],
       [3, 7, 8, 9]])

In [202]:
z = np.arange(1, 13).reshape((3, 4))

z

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

In [204]:
np.dstack((z, x3))

array([[[ 1,  1,  8,  7,  0,  6],
        [ 2,  2,  9,  5,  5,  6],
        [ 3,  0,  6,  0,  4,  6],
        [ 4,  3,  5,  7,  9,  2]],

       [[ 5,  9,  3,  3,  6,  9],
        [ 6,  3,  4,  8,  5,  5],
        [ 7,  3,  3,  0,  0,  4],
        [ 8,  8,  7,  5,  2,  9]],

       [[ 9,  1,  8,  0,  5,  1],
        [10,  2,  2,  6,  8,  4],
        [11,  9,  4,  1,  5,  3],
        [12,  7,  8,  3,  8,  0]]])

In [205]:
x3.shape, z.shape

((3, 4, 5), (3, 4))

> Similarly, for higher-dimensional arrays, **np.dstack** will stack arrays along the third axis.

### **Splitting of Arrays**

In [208]:
x = [1, 2, 3, 99, 99, 3, 2, 1]
x1, x2, x3 = np.split(x, [3, 5])

print(x1, x2, x3)

[1 2 3] [99 99] [3 2 1]


In [212]:
x1, x2, x3 = np.split(x, [2, 6])

print(x1, x2, x3)

[1 2] [ 3 99 99  3] [2 1]


In [213]:
grid = np.arange(16).reshape((4, 4))

grid

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15]])

In [214]:
np.vsplit(grid, 2)

[array([[0, 1, 2, 3],
        [4, 5, 6, 7]]),
 array([[ 8,  9, 10, 11],
        [12, 13, 14, 15]])]

In [216]:
np.vsplit(grid, [3, 6])

[array([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]]),
 array([[12, 13, 14, 15]]),
 array([], shape=(0, 4), dtype=int64)]

In [217]:
np.hsplit(grid, 2)

[array([[ 0,  1],
        [ 4,  5],
        [ 8,  9],
        [12, 13]]),
 array([[ 2,  3],
        [ 6,  7],
        [10, 11],
        [14, 15]])]

In [218]:
np.split(grid, [3, 6])

[array([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]]),
 array([[12, 13, 14, 15]]),
 array([], shape=(0, 4), dtype=int64)]

In [229]:
x3 = np.arange(60).reshape((3, 4, 5))

x3.shape

(3, 4, 5)

In [234]:
x3

array([[[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]],

       [[20, 21, 22, 23, 24],
        [25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34],
        [35, 36, 37, 38, 39]],

       [[40, 41, 42, 43, 44],
        [45, 46, 47, 48, 49],
        [50, 51, 52, 53, 54],
        [55, 56, 57, 58, 59]]])

In [235]:
np.dsplit(x3, [2, 3])

[array([[[ 0,  1],
         [ 5,  6],
         [10, 11],
         [15, 16]],
 
        [[20, 21],
         [25, 26],
         [30, 31],
         [35, 36]],
 
        [[40, 41],
         [45, 46],
         [50, 51],
         [55, 56]]]),
 array([[[ 2],
         [ 7],
         [12],
         [17]],
 
        [[22],
         [27],
         [32],
         [37]],
 
        [[42],
         [47],
         [52],
         [57]]]),
 array([[[ 3,  4],
         [ 8,  9],
         [13, 14],
         [18, 19]],
 
        [[23, 24],
         [28, 29],
         [33, 34],
         [38, 39]],
 
        [[43, 44],
         [48, 49],
         [53, 54],
         [58, 59]]])]

60